In [ ]:
from pathlib import Path

import seaborn as sns
from bonner.plotting import save_figure
from matplotlib import pyplot as plt
from tqdm.auto import tqdm

from lib.datasets import (
    compute_shared_stimuli,
    filter_by_stimulus,
    nsd,
    split_by_repetition,
)
from lib.spectra import (
    compute_within_individual_spectra,
    plot_spectra,
)
from lib.utilities import (
    JOURNAL_MATPLOTLIBRC,
    mathtext_exponent_label,
)

FIGURES_HOME = Path.cwd().parent / "figures"
FIGURES_HOME.mkdir(exist_ok=True, parents=True)

sns.set_theme(context="paper", style="ticks", rc=JOURNAL_MATPLOTLIBRC)

REFERENCE_SUBJECT = 0

ROIS = ("faces", "bodies", "places", "words")

In [ ]:
datasets = {
    roi: {
        subject: nsd.load_dataset(
            subject=subject,
            roi=roi,
            preprocessing="fithrf",
            z_score=True,
        )
        for subject in range(nsd.N_SUBJECTS)
    }
    for roi in ROIS
}

datasets_within = {
    roi: {
        subject: split_by_repetition(
            filter_by_stimulus(
                dataset,
                stimuli=compute_shared_stimuli([dataset], n_repetitions=2),
            ),
            n_repetitions=2,
        )
        for subject, dataset in datasets_.items()
    }
    for roi, datasets_ in datasets.items()
}

In [ ]:
spectra_within = {
    roi: compute_within_individual_spectra(
        datasets_within[roi],
        n_permutations=5_000,
    ).expand_dims({"region of interest": [roi]})
    for roi in tqdm(ROIS, desc="region of interest", leave=False)
}

In [ ]:
fig, axes = plt.subplots(figsize=(6, 2.5), ncols=4, sharex=True, sharey=True)

kwargs_legend = {
    "loc": "upper right",
    "title": "subject",
    "ncols": 2,
    "columnspacing": 0.25,
    "handletextpad": 0,
    "reverse": True,
    "borderaxespad": 0,
    "borderpad": 0,
}

for i_roi, roi in enumerate(ROIS):
    ax = axes[i_roi]
    plot_spectra(
        ax=ax,
        spectra=spectra_within[roi],
        hue="individual",
        palette="crest",
        hue_order=list(reversed(range(nsd.N_SUBJECTS))),
        hue_labels=[f"{subject + 1}" for subject in reversed(range(nsd.N_SUBJECTS))],
        marker="s",
        hide_insignificant=True,
        null_quantile=0.999,
    )
    ax.set_title(roi)

    ax.set_xscale("log")
    ax.set_yscale("log")
    ax.set_xlim(left=1, right=1e4)
    ax.set_xticks([1, 1e1, 1e2, 1e3, 1e4])
    ax.set_ylim(top=1e-1, bottom=1e-7)
    ytick_exponents = list(range(-7, 0))
    ax.set_yticks(
        [10**exponent for exponent in ytick_exponents],
        labels=[
            mathtext_exponent_label(exponent) if exponent % 2 == 1 else ""
            for exponent in ytick_exponents
        ],
    )

axes[0].set_ylabel("covariance")
fig.supxlabel("rank", x=0.54, y=0.05)
fig.suptitle("high-level category-selective regions")

axes[-1].legend(**kwargs_legend)
save_figure(fig, filepath=FIGURES_HOME / "category-selective.pdf")